<a href="https://colab.research.google.com/github/Spoorti803/Deep_Reinforcement_Learning/blob/main/PPO_LunarLander_V3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q "gymnasium[box2d]" stable-baselines3

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 57.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 91.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 187.6/187.6 kB 21.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.1/12.1 MB 101.5 MB/s eta 0:00:00


In [ ]:
"""
gymnasium: provides the LunarLander environment.
stable-baselines3: provides the PPO algorithm and neural-network policy implementation.
"""

In [2]:
import gymnasium as gym
env = gym.make('LunarLander-v3')

observation, info = env.reset()

print("Initial observation: ",observation)
print("Observation shape: ",observation.shape)
print("Action space:",env.action_space)


Initial observation:  [-0.00146799  1.4056246  -0.14870402 -0.23535201  0.00170778  0.03368365
  0.          0.        ]
Observation shape:  (8,)
Action space: Discrete(4)


You should see an observation shape of (8,) and an action space similar to Discrete(4).<br>
The four actions are:<br>
1. Do nothing
2. Fire left orientation engine
3. Fire main engine
4. Fire right orientation engine

In [3]:
#before training the PPO let us check what happens when randomly actions are taken
for step in range(20):
  action = env.action_space.sample()

  observation, reward, terminated, truncated, info = env.step(action)
  print("Step: ",step)
  print("Action: ",action)
  print("Reward: ",reward)

  if terminated or truncated:
    observation, info = env.reset()

env.close()


Step:  0
Action:  2
Reward:  0.6403125608140499
Step:  1
Action:  1
Reward:  -2.73576971829459
Step:  2
Action:  3
Reward:  -1.5461411821052582
Step:  3
Action:  3
Reward:  -1.2370549555831747
Step:  4
Action:  3
Reward:  -1.3095315934705514
Step:  5
Action:  2
Reward:  4.4730236023730585
Step:  6
Action:  1
Reward:  -2.1202543263624123
Step:  7
Action:  2
Reward:  2.486375654919141
Step:  8
Action:  2
Reward:  2.850794901272292
Step:  9
Action:  1
Reward:  -2.6257190690403704
Step:  10
Action:  1
Reward:  -2.7019538500491067
Step:  11
Action:  0
Reward:  -2.328912390678937
Step:  12
Action:  1
Reward:  -2.8838482545390023
Step:  13
Action:  3
Reward:  -1.9185110664759282
Step:  14
Action:  1
Reward:  -2.7108629976604264
Step:  15
Action:  2
Reward:  3.865970801741452
Step:  16
Action:  2
Reward:  -0.2222681056184001
Step:  17
Action:  1
Reward:  -3.05193152154328
Step:  18
Action:  0
Reward:  -2.5935728576938573
Step:  19
Action:  2
Reward:  0.9108582634121489


`env.step(action)` applies the choosen action and returns 5 things:
1. observation: the new state.
2. reward: the feedback received.
3. terminated: whether the episode ended naturally, for example through landing or crashing.
4. truncated: whether the episode stopped because of a limit, such as the time limit.
5. info: additional diagnostic information.

### Train the Deep RL

In [5]:
#create the PPO model

from stable_baselines3 import PPO

env = gym.make('LunarLander-v3')
"""
Mlp policy = mulitlayer perceptron policy
env = connects the model to the environment
verbose = 1 displays the training information
"""
model = PPO("MlpPolicy",env,verbose=1)

Using cuda device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.


## Result-1

In [6]:
model.learn(total_timesteps=20_000)
model.save("ppo_Lunarlander-v3")
print("training finished and model saved")

---------------------------------
| rollout/           |          |
|    ep_len_mean     | 85.3     |
|    ep_rew_mean     | -187     |
| time/              |          |
|    fps             | 582      |
|    iterations      | 1        |
|    time_elapsed    | 3        |
|    total_timesteps | 2048     |
---------------------------------
-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 90.5        |
|    ep_rew_mean          | -185        |
| time/                   |             |
|    fps                  | 464         |
|    iterations           | 2           |
|    time_elapsed         | 8           |
|    total_timesteps      | 4096        |
| train/                  |             |
|    approx_kl            | 0.009402078 |
|    clip_fraction        | 0.0642      |
|    clip_range           | 0.2         |
|    entropy_loss         | -1.38       |
|    explained_variance   | 0.00751     |
|    learning_rate        | 0.

In [7]:
import gymnasium as gym
from gymnasium.wrappers import RecordVideo
from IPython.display import Video, display
import os

eval_env = gym.make(
    "LunarLander-v3",
    render_mode="rgb_array"
)

eval_env = RecordVideo(
    eval_env,
    video_folder="videos",
    episode_trigger=lambda episode_id: True
)

observation, info = eval_env.reset()
terminated = False
truncated = False
total_reward = 0

while not (terminated or truncated):
    action, _ = model.predict(
        observation,
        deterministic=True
    )

    observation, reward, terminated, truncated, info = (
        eval_env.step(action)
    )

    total_reward += reward

eval_env.close()

print("Total episode reward:", round(total_reward, 2))

video_files = [
    f for f in os.listdir("videos")
    if f.endswith(".mp4")
]

if video_files:
    display(Video(
        os.path.join("videos", video_files[-1]),
        embed=True
    ))

/usr/local/lib/python3.13/dist-packages/moviepy/config_defaults.py:47: SyntaxWarning: invalid escape sequence '\P'
  IMAGEMAGICK_BINARY = r"C:\Program Files\ImageMagick-6.8.8-Q16\magick.exe"
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


Total episode reward: -1183.09


## Result-2



In [9]:
model.learn(total_timesteps=200_000)
model.save("ppo_Lunarlander-v3")
print("training finished and model saved")

---------------------------------
| rollout/           |          |
|    ep_len_mean     | 133      |
|    ep_rew_mean     | -123     |
| time/              |          |
|    fps             | 631      |
|    iterations      | 1        |
|    time_elapsed    | 3        |
|    total_timesteps | 2048     |
---------------------------------
------------------------------------------
| rollout/                |              |
|    ep_len_mean          | 178          |
|    ep_rew_mean          | -98.7        |
| time/                   |              |
|    fps                  | 523          |
|    iterations           | 2            |
|    time_elapsed         | 7            |
|    total_timesteps      | 4096         |
| train/                  |              |
|    approx_kl            | 0.0097501185 |
|    clip_fraction        | 0.0741       |
|    clip_range           | 0.2          |
|    entropy_loss         | -1.14        |
|    explained_variance   | 0.0644       |
|    learning_r

In [10]:
import gymnasium as gym
from gymnasium.wrappers import RecordVideo
from IPython.display import Video, display
import os

eval_env = gym.make(
    "LunarLander-v3",
    render_mode="rgb_array"
)

eval_env = RecordVideo(
    eval_env,
    video_folder="videos",
    episode_trigger=lambda episode_id: True
)

observation, info = eval_env.reset()
terminated = False
truncated = False
total_reward = 0

while not (terminated or truncated):
    action, _ = model.predict(
        observation,
        deterministic=True
    )

    observation, reward, terminated, truncated, info = (
        eval_env.step(action)
    )

    total_reward += reward

eval_env.close()

print("Total episode reward:", round(total_reward, 2))

video_files = [
    f for f in os.listdir("videos")
    if f.endswith(".mp4")
]

if video_files:
    display(Video(
        os.path.join("videos", video_files[-1]),
        embed=True
    ))

/usr/local/lib/python3.13/dist-packages/gymnasium/wrappers/rendering.py:292: UserWarning: WARN: Overwriting existing videos at /content/videos folder (try specifying a different `video_folder` for the `RecordVideo` wrapper if this is not desired)
  logger.warn(


Total episode reward: 217.53


In [ ]:
#train the model

model.learn(total_timesteps=500_000)
model.save("ppo_Lunarlander-v3")
print("training finished and model saved")

---------------------------------
| rollout/           |          |
|    ep_len_mean     | 88.1     |
|    ep_rew_mean     | -175     |
| time/              |          |
|    fps             | 555      |
|    iterations      | 1        |
|    time_elapsed    | 3        |
|    total_timesteps | 2048     |
---------------------------------
-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 87.6        |
|    ep_rew_mean          | -173        |
| time/                   |             |
|    fps                  | 433         |
|    iterations           | 2           |
|    time_elapsed         | 9           |
|    total_timesteps      | 4096        |
| train/                  |             |
|    approx_kl            | 0.005466336 |
|    clip_fraction        | 0.0041      |
|    clip_range           | 0.2         |
|    entropy_loss         | -1.38       |
|    explained_variance   | -0.00268    |
|    learning_rate        | 0.

In [ ]:
import gymnasium as gym
from gymnasium.wrappers import RecordVideo
from IPython.display import Video, display
import os

eval_env = gym.make(
    "LunarLander-v3",
    render_mode="rgb_array"
)

eval_env = RecordVideo(
    eval_env,
    video_folder="videos",
    episode_trigger=lambda episode_id: True
)

observation, info = eval_env.reset()
terminated = False
truncated = False
total_reward = 0

while not (terminated or truncated):
    action, _ = model.predict(
        observation,
        deterministic=True
    )

    observation, reward, terminated, truncated, info = (
        eval_env.step(action)
    )

    total_reward += reward

eval_env.close()

print("Total episode reward:", round(total_reward, 2))

video_files = [
    f for f in os.listdir("videos")
    if f.endswith(".mp4")
]

if video_files:
    display(Video(
        os.path.join("videos", video_files[-1]),
        embed=True
    ))

/usr/local/lib/python3.13/dist-packages/moviepy/config_defaults.py:47: SyntaxWarning: invalid escape sequence '\P'
  IMAGEMAGICK_BINARY = r"C:\Program Files\ImageMagick-6.8.8-Q16\magick.exe"
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


Total episode reward: 124.28


In [ ]:
import gymnasium as gym
import numpy as np

eval_env = gym.make("LunarLander-v3")

rewards = []

for episode in range(10):
    obs, info = eval_env.reset()
    total_reward = 0
    terminated = False
    truncated = False

    while not (terminated or truncated):
        action, _ = model.predict(
            obs, deterministic=True
        )

        obs, reward, terminated, truncated, info = (
            eval_env.step(action)
        )

        total_reward += reward

    rewards.append(total_reward)
    print(f"Episode {episode + 1}: {total_reward:.2f}")

eval_env.close()

print("Average reward:", np.mean(rewards))

Episode 1: 267.61
Episode 2: 239.81
Episode 3: 258.06
Episode 4: 258.46
Episode 5: 262.01
Episode 6: 231.24
Episode 7: 294.95
Episode 8: 313.63
Episode 9: 275.47
Episode 10: 275.29
Average reward: 267.653815914295
